# Ridge Regression from Scratch

## Introduction

**Ridge Regression** (also known as **L2 Regularization** or **Tikhonov Regularization**) is a technique used to address **overfitting** in linear regression by adding a penalty term to the cost function.

---

## Why Ridge Regression?

Standard Linear Regression minimizes:
$$J(\theta) = ||y - X\theta||^2$$

**Problems with OLS (Ordinary Least Squares):**
- Overfitting when features > samples
- Multicollinearity causes unstable coefficients
- High variance in predictions

**Ridge Regression** adds an L2 penalty:
$$J(\theta) = ||y - X\theta||^2 + \lambda||\theta||^2$$

Where:
- $\lambda$ (alpha) = regularization strength
- $||\theta||^2 = \sum_{j=1}^{n} \theta_j^2$ = sum of squared coefficients

---

## Mathematical Derivation

### The Ridge Cost Function

$$J(\theta) = (y - X\theta)^T(y - X\theta) + \lambda \theta^T\theta$$

### Taking the Gradient

$$\frac{\partial J}{\partial \theta} = -2X^T(y - X\theta) + 2\lambda\theta$$

### Setting Gradient to Zero

$$-2X^T(y - X\theta) + 2\lambda\theta = 0$$
$$X^Ty - X^TX\theta + \lambda\theta = 0$$
$$X^Ty = X^TX\theta - \lambda\theta$$
$$X^Ty = (X^TX + \lambda I)\theta$$

### Closed-Form Solution

$$\boxed{\theta = (X^TX + \lambda I)^{-1}X^Ty}$$

This is the **Ridge Regression closed-form solution**!

---

## Import Libraries

In [1]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score,mean_squared_error
import numpy as np

---

## Load and Explore Dataset

We'll use the **Diabetes dataset** from sklearn, which has 10 features and 442 samples.

In [3]:
X,y = load_diabetes(return_X_y=True)

In [20]:
X.shape

(442, 10)

In [16]:
y.shape

(442,)

In [4]:
from sklearn.model_selection import train_test_split

#### `random_state` Parameter in `train_test_split`
The `random_state` parameter controls the randomness of the data splitting:

What it does:
- `train_test_split` randomly shuffles data before splitting into train/test sets
- `random_state` sets the seed for the random number generator
- Using the same value (e.g., `random_state=4`) ensures you get identical splits every time you run the code


In [9]:

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=4)

---

## Part 1: Ridge Regression using Scikit-Learn

First, let's implement Ridge Regression using sklearn's built-in `Ridge` class to establish a baseline.

### Ridge Parameters:
- **alpha**: Regularization strength ($\lambda$). Larger values = stronger regularization
- **solver**: Algorithm to solve the optimization problem
  - `"cholesky"`: Uses Cholesky decomposition (closed-form solution)

In [10]:
from sklearn.linear_model import Ridge

In [11]:
Ridge_regressor_model = Ridge(alpha=0.1,solver="cholesky")

In [12]:
Ridge_regressor_model.fit(X_train,y_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,alpha,0.1
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'cholesky'
,positive,False
,random_state,None


In [21]:
print(Ridge_regressor_model.coef_)
print(Ridge_regressor_model.intercept_)

[  42.52213989 -230.953097    420.64961039  357.28171241  -75.62837476
  -59.05156776 -175.8466958   146.1687604   418.11706965   62.54065064]
151.20369941010867


---

## Part 2: Ridge Regression from Scratch

Now let's implement Ridge Regression from scratch using the closed-form solution:

$$\theta = (X^TX + \lambda I)^{-1}X^Ty$$

---

### Important: Why We Set `I[0][0] = 0`

When implementing Ridge Regression with an **intercept (bias) term**, we need to handle it carefully.

#### The Setup

We add a column of 1s to X for the intercept:
```
X_augmented = [1, x₁, x₂, ..., xₙ]
```

This means our parameter vector becomes:
```
θ = [θ₀, θ₁, θ₂, ..., θₙ]
     ↑
   intercept
```

#### The Problem

The Ridge penalty is:
$$\lambda \sum_{j=0}^{n} \theta_j^2 = \lambda(\theta_0^2 + \theta_1^2 + ... + \theta_n^2)$$

But **we should NOT regularize the intercept** $\theta_0$!

#### Why Not Regularize the Intercept?

1. **The intercept represents the baseline prediction** — it shifts the entire prediction up/down
2. **Regularizing it would bias predictions toward zero** unnecessarily
3. **It doesn't contribute to overfitting** — only the feature coefficients do
4. **Standard practice** — sklearn and all ML libraries exclude intercept from regularization

#### The Solution

We modify the identity matrix I:

**Before:**
$$I = \begin{bmatrix} 1 & 0 & 0 & \cdots \\ 0 & 1 & 0 & \cdots \\ 0 & 0 & 1 & \cdots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}$$

**After setting `I[0][0] = 0`:**
$$I_{modified} = \begin{bmatrix} \boxed{0} & 0 & 0 & \cdots \\ 0 & 1 & 0 & \cdots \\ 0 & 0 & 1 & \cdots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}$$

This makes the penalty:
$$\lambda \cdot I_{modified} \cdot \theta = \lambda(0 \cdot \theta_0^2 + 1 \cdot \theta_1^2 + 1 \cdot \theta_2^2 + ...)$$

**The intercept is now excluded from regularization!**

---

### Custom Ridge Regression Class

In [22]:
class MeraRidge:
    
    def __init__(self,alpha=0.1):
        
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self,X_train,y_train):
        
        X_train = np.insert(X_train,0,1,axis=1)
        I = np.identity(X_train.shape[1])
        I[0][0] = 0
        result = np.linalg.inv(np.dot(X_train.T,X_train) + self.alpha * I).dot(X_train.T).dot(y_train)
        self.intercept_ = result[0]
        self.coef_ = result[1:]
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_

#### Code Explanation

```python
# Step 1: Add intercept column (column of 1s at position 0)
X_train = np.insert(X_train, 0, 1, axis=1)

# Step 2: Create identity matrix of size (n_features + 1)
I = np.identity(X_train.shape[1])

# Step 3: CRITICAL - Don't regularize the intercept!
I[0][0] = 0

# Step 4: Apply closed-form formula
# θ = (X'X + λI)⁻¹ X'y
result = np.linalg.inv(X_train.T @ X_train + alpha * I) @ X_train.T @ y_train

# Step 5: Extract intercept (first element) and coefficients (rest)
intercept = result[0]
coefficients = result[1:]
```

---

### Prediction Formula Explained

```python
def predict(self, X_test):
    return np.dot(X_test, self.coef_) + self.intercept_
```

#### Mathematical Representation

The prediction formula is:

$$\hat{y} = X \cdot \beta + \beta_0$$

Or expanded for a single sample:

$$\hat{y}_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + ... + \beta_n x_{in}$$

Where:
- $\hat{y}$ = predicted values (vector of shape `(n_samples,)`)
- $X$ = feature matrix (shape `(n_samples, n_features)`)
- $\beta$ = coefficient vector (`self.coef_`, shape `(n_features,)`)
- $\beta_0$ = intercept (`self.intercept_`, scalar)

#### Step-by-Step Breakdown

| Code | Operation | Result Shape |
|------|-----------|--------------|
| `X_test` | Input features | `(n_samples, n_features)` |
| `self.coef_` | Learned coefficients | `(n_features,)` |
| `np.dot(X_test, self.coef_)` | Matrix-vector multiplication | `(n_samples,)` |
| `+ self.intercept_` | Add bias term (broadcasts) | `(n_samples,)` |

#### Visual Example

For a single sample with 3 features:

$$\hat{y} = \underbrace{\begin{bmatrix} x_1 & x_2 & x_3 \end{bmatrix}}_{X_{test}} \cdot \underbrace{\begin{bmatrix} \beta_1 \\ \beta_2 \\ \beta_3 \end{bmatrix}}_{\text{coef\_}} + \underbrace{\beta_0}_{\text{intercept\_}}$$

$$= x_1 \beta_1 + x_2 \beta_2 + x_3 \beta_3 + \beta_0$$

#### Why This Works

During training, we computed:
$$\theta = (X^TX + \lambda I)^{-1}X^Ty = \begin{bmatrix} \beta_0 \\ \beta_1 \\ \vdots \\ \beta_n \end{bmatrix}$$

We then separated:
- `intercept_ = result[0]` → $\beta_0$
- `coef_ = result[1:]` → $[\beta_1, \beta_2, ..., \beta_n]$

For prediction, we reconstruct:
$$\hat{y} = X_{test} \cdot \text{coef\_} + \text{intercept\_}$$

This is equivalent to the original formulation where we had added a column of 1s to X.

---

### Test Our Custom Implementation

In [23]:
reg = MeraRidge()
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
print(r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

0.46734572740930014
[  42.52213989 -230.953097    420.64961039  357.28171241  -75.62837476
  -59.05156776 -175.8466958   146.1687604   418.11706965   62.54065064]
151.2036994101087


---

## Comparison: Sklearn vs Our Implementation

Both implementations should produce **identical results** because they use the same closed-form solution:

| Metric | Sklearn Ridge | MeraRidge (Custom) |
|--------|---------------|-------------------|
| R² Score | ✓ | ✓ |
| Coefficients | ✓ | ✓ |
| Intercept | ✓ | ✓ |

---

## Summary

### Key Formulas

| Component | Formula |
|-----------|---------|
| **Cost Function** | $J(\theta) = \|y - X\theta\|^2 + \lambda\|\theta\|^2$ |
| **Closed-Form Solution** | $\theta = (X^TX + \lambda I)^{-1}X^Ty$ |
| **Prediction** | $\hat{y} = X\theta = X\beta + \theta_0$ |

### Key Insights

1. **Ridge adds L2 penalty** to shrink coefficients toward zero (but not exactly zero)

2. **`I[0][0] = 0`** ensures the intercept is NOT regularized

3. **Alpha (λ) controls regularization strength**:
   - α → 0: Behaves like regular linear regression
   - α → ∞: All coefficients shrink toward zero

4. **Ridge is always invertible**: Adding λI to X'X guarantees invertibility even when X'X is singular

---

### When to Use Ridge Regression?

| Scenario | Use Ridge? |
|----------|------------|
| Many correlated features | ✅ Yes |
| Overfitting | ✅ Yes |
| Need feature selection | ❌ No (use Lasso) |
| All features likely important | ✅ Yes |